# CONFIG

In [0]:
# Databricks notebook source
from pyspark.sql import functions as F
from pyspark.sql import types as T

# =========================
# 0) CONFIG
# =========================
BRZ = "cms_partd_bronze.raw"
SLV = "cms_partd_silver.core"

# If you use a contract_year column in bronze, set it here; otherwise we keep as-is
DEFAULT_CONTRACT_YEAR = None  # e.g. "2025"

# =========================
# 1) HELPER FUNCTIONS
# =========================

def to_snake(s: str) -> str:
    # simple snake conversion; keeps digits; replaces spaces and punctuation with underscore
    import re
    s = s.strip()
    s = re.sub(r"[^\w]+", "_", s)     # non-word -> _
    s = re.sub(r"([a-z0-9])([A-Z])", r"\1_\2", s)  # camel -> snake
    s = re.sub(r"_+", "_", s)
    return s.lower()

def normalize_string(col):
    return F.trim(F.col(col)).cast("string")

def normalize_id(col, pad_left=None):
    c = F.upper(F.trim(F.col(col).cast("string")))
    if pad_left is not None:
        c = F.lpad(c, pad_left, "0")
    return c

def cast_decimal(col, precision, scale):
    return F.col(col).cast(T.DecimalType(precision, scale))

def add_ingest_metadata(df):
    # keep bronze metadata if present; otherwise add
    if "ingest_ts" not in df.columns:
        df = df.withColumn("ingest_ts", F.current_timestamp())
    if "ingest_date" not in df.columns:
        df = df.withColumn("ingest_date", F.to_date(F.current_timestamp()))
    if "source_file" not in df.columns:
        df = df.withColumn("source_file", F.lit(None).cast("string"))
    return df

def standardize_columns(df):
    # rename all columns to snake_case
    for c in df.columns:
        df = df.withColumnRenamed(c, to_snake(c))
    return df

# Days supply mapping for Beneficiary Cost (coded) vs Pricing (numeric)
# Beneficiary cost DAYS_SUPPLY: 1=30, 2=90, 3=other, 4=60
benef_days_supply_label = (
    F.when(F.col("days_supply") == 1, F.lit("30"))
     .when(F.col("days_supply") == 2, F.lit("90"))
     .when(F.col("days_supply") == 4, F.lit("60"))
     .when(F.col("days_supply") == 3, F.lit("OTHER"))
     .otherwise(F.lit(None))
)

cost_type_label = (
    F.when(F.col("x") == 0, F.lit("NOT_OFFERED"))
     .when(F.col("x") == 1, F.lit("COPAY"))
     .when(F.col("x") == 2, F.lit("COINSURANCE"))
     .otherwise(F.lit(None))
)

def safe_bool_from_yn(colname):
    # Y/N -> boolean; handles null
    return (
        F.when(F.upper(F.trim(F.col(colname))) == F.lit("Y"), F.lit(True))
         .when(F.upper(F.trim(F.col(colname))) == F.lit("N"), F.lit(False))
         .otherwise(F.lit(None).cast("boolean"))
    )

def safe_int_from_flag(colname):
    # '1'/'0' or int -> int
    return F.col(colname).cast("int")


def flag_to_int(c):
    s = F.upper(F.trim(F.col(c).cast("string")))
    return (
        F.when(s.isNull() | (s == ""), F.lit(None).cast("int"))
         .when(s.isin("Y", "YES", "T", "TRUE", "1"), F.lit(1))
         .when(s.isin("N", "NO", "F", "FALSE", "0"), F.lit(0))
         .otherwise(F.expr(f"try_cast({c} as int)"))  # tolerate weird cases
    )



# SILVER: PLAN

In [0]:
# =========================
# 2) SILVER: PLAN
# =========================
df_plan_brz = spark.table(f"{BRZ}.brz_plan_info")
df_plan = (
    df_plan_brz
    .transform(standardize_columns)
    .transform(add_ingest_metadata)
    # Normalize IDs
    .withColumn("contract_id", normalize_id("contract_id"))
    .withColumn("plan_id", normalize_id("plan_id", pad_left=3))
    .withColumn("segment_id", normalize_id("segment_id", pad_left=3))
    .withColumn("formulary_id", normalize_id("formulary_id"))
    .withColumn("state", normalize_id("state", pad_left=2))
    .withColumn("county_code", normalize_id("county_code", pad_left=5))
    # Money fields
    .withColumn("premium", cast_decimal("premium", 12, 2))
    .withColumn("deductible", cast_decimal("deductible", 12, 2))
    # Derived
    .withColumn("contract_type", F.substring(F.col("contract_id"), 1, 1))  # H/R/S
    .withColumn(
        "plan_bk",
        F.concat_ws("|", F.col("contract_id"), F.col("plan_id"), F.col("segment_id"))
    )
)

# If you have plan_key in bronze, keep it. If not, create a stable surrogate key hash
if "plan_key" not in df_plan.columns:
    df_plan = df_plan.withColumn("plan_key", F.xxhash64("plan_bk"))

# Optional: contract_year
if DEFAULT_CONTRACT_YEAR and "contract_year" not in df_plan.columns:
    df_plan = df_plan.withColumn("contract_year", F.lit(DEFAULT_CONTRACT_YEAR))

df_plan.write.format("delta").mode("overwrite").saveAsTable(f"{SLV}.slv_plan")


# SILVER: GEOGRAPHY

In [0]:

# =========================
# 3) SILVER: GEOGRAPHY
# =========================
df_geo_brz = spark.table(f"{BRZ}.brz_geographic_locator")
df_geo = (
    df_geo_brz
    .transform(standardize_columns)
    .transform(add_ingest_metadata)
    .withColumn("county_code", normalize_id("county_code", pad_left=5))
    .withColumn("statename", normalize_string("statename"))
    .withColumn("county", normalize_string("county"))
    .withColumn("ma_region_code", normalize_id("ma_region_code", pad_left=2))
    .withColumn("pdp_region_code", normalize_id("pdp_region_code", pad_left=2))
)

df_geo.write.format("delta").mode("overwrite").saveAsTable(f"{SLV}.slv_geo")


# SILVER: BENEFICIARY COST

In [0]:

# =========================
# 4) SILVER: BENEFICIARY COST
# =========================
df_cost_brz = spark.table(f"{BRZ}.brz_beneficiary_cost")
df_cost = (
    df_cost_brz
    .transform(standardize_columns)
    .transform(add_ingest_metadata)
    # Normalize plan identifiers
    .withColumn("contract_id", normalize_id("contract_id"))
    .withColumn("plan_id", normalize_id("plan_id", pad_left=3))
    .withColumn("segment_id", normalize_id("segment_id", pad_left=3))
    .withColumn("plan_bk", F.concat_ws("|", "contract_id", "plan_id", "segment_id"))
    # Codes
    .withColumn("coverage_level", F.col("coverage_level").cast("int"))
    .withColumn("tier", F.col("tier").cast("int"))
    .withColumn("days_supply", F.col("days_supply").cast("int"))
    .withColumn("days_supply_label", benef_days_supply_label)
    # Cost types
    .withColumn("cost_type_pref", F.col("cost_type_pref").cast("int"))
    .withColumn("cost_type_nonpref", F.col("cost_type_nonpref").cast("int"))
    .withColumn("cost_type_mail_pref", F.col("cost_type_mail_pref").cast("int"))
    .withColumn("cost_type_mail_nonpref", F.col("cost_type_mail_nonpref").cast("int"))
    # Labels (avoid repeating logic)
    .withColumn("cost_type_pref_label", F.expr("case cost_type_pref when 0 then 'NOT_OFFERED' when 1 then 'COPAY' when 2 then 'COINSURANCE' end"))
    .withColumn("cost_type_nonpref_label", F.expr("case cost_type_nonpref when 0 then 'NOT_OFFERED' when 1 then 'COPAY' when 2 then 'COINSURANCE' end"))
    .withColumn("cost_type_mail_pref_label", F.expr("case cost_type_mail_pref when 0 then 'NOT_OFFERED' when 1 then 'COPAY' when 2 then 'COINSURANCE' end"))
    .withColumn("cost_type_mail_nonpref_label", F.expr("case cost_type_mail_nonpref when 0 then 'NOT_OFFERED' when 1 then 'COPAY' when 2 then 'COINSURANCE' end"))
    # Amounts
    .withColumn("cost_amt_pref", cast_decimal("cost_amt_pref", 12, 2))
    .withColumn("cost_max_amt_pref", cast_decimal("cost_max_amt_pref", 12, 2))
    .withColumn("cost_amt_nonpref", cast_decimal("cost_amt_nonpref", 12, 2))
    .withColumn("cost_max_amt_nonpref", cast_decimal("cost_max_amt_nonpref", 12, 2))
    .withColumn("cost_amt_mail_pref", cast_decimal("cost_amt_mail_pref", 12, 2))
    .withColumn("cost_max_amt_mail_pref", cast_decimal("cost_max_amt_mail_pref", 12, 2))
    .withColumn("cost_amt_mail_nonpref", cast_decimal("cost_amt_mail_nonpref", 12, 2))
    .withColumn("cost_max_amt_mail_nonpref", cast_decimal("cost_max_amt_mail_nonpref", 12, 2))
    # Min amounts sometimes come as string/char (from spec). Keep as decimal when parseable, else null.
    .withColumn("cost_min_amt_pref", F.regexp_replace(F.col("cost_min_amt_pref").cast("string"), r"[^0-9\.\-]", "").cast(T.DecimalType(12,2)))
    .withColumn("cost_min_amt_nonpref", F.regexp_replace(F.col("cost_min_amt_nonpref").cast("string"), r"[^0-9\.\-]", "").cast(T.DecimalType(12,2)))
    .withColumn("cost_min_amt_mail_pref", F.regexp_replace(F.col("cost_min_amt_mail_pref").cast("string"), r"[^0-9\.\-]", "").cast(T.DecimalType(12,2)))
    .withColumn("cost_min_amt_mail_nonpref", F.regexp_replace(F.col("cost_min_amt_mail_nonpref").cast("string"), r"[^0-9\.\-]", "").cast(T.DecimalType(12,2)))
    # Flags
    .withColumn("tier_specialty_yn", safe_bool_from_yn("tier_specialty_yn") if "tier_specialty_yn" in df_cost_brz.columns else F.col("tier_specialty_yn"))
    .withColumn("ded_applies_yn", safe_bool_from_yn("ded_applies_yn") if "ded_applies_yn" in df_cost_brz.columns else F.col("ded_applies_yn"))
)

# Attach plan_key if it exists in bronze, else join from slv_plan
df_plan_keys = spark.table(f"{SLV}.slv_plan").select("plan_key", "plan_bk")
if "plan_key" not in df_cost.columns:
    df_cost = df_cost.join(df_plan_keys, on="plan_bk", how="left")

# Create deterministic cost grain key if not present
if "cost_key" not in df_cost.columns:
    df_cost = df_cost.withColumn(
        "cost_key",
        F.xxhash64("plan_key", "coverage_level", "tier", "days_supply")
    )

df_cost.write.format("delta").mode("overwrite").saveAsTable(f"{SLV}.slv_beneficiary_cost")


# SILVER: PRICING

In [0]:

# =========================
# 5) SILVER: PRICING
# =========================
df_pricing_brz = spark.table(f"{BRZ}.brz_pricing")
df_pricing = (
    df_pricing_brz
    .transform(standardize_columns)
    .transform(add_ingest_metadata)
    .withColumn("contract_id", normalize_id("contract_id"))
    .withColumn("plan_id", normalize_id("plan_id", pad_left=3))
    .withColumn("segment_id", normalize_id("segment_id", pad_left=3))
    .withColumn("plan_bk", F.concat_ws("|", "contract_id", "plan_id", "segment_id"))
    .withColumn("ndc", normalize_id("ndc", pad_left=11))
    .withColumn("days_supply", F.col("days_supply").cast("int"))  # should be 30/60/90 per spec
    .withColumn("unit_cost", F.col("unit_cost").cast(T.DecimalType(12,4)))
)

# Add plan_key
if "plan_key" not in df_pricing.columns:
    df_pricing = df_pricing.join(df_plan_keys, on="plan_bk", how="left")

# Create pricing_key if absent
if "pricing_key" not in df_pricing.columns:
    df_pricing = df_pricing.withColumn("pricing_key", F.xxhash64("plan_key", "ndc", "days_supply"))

df_pricing.write.format("delta").mode("overwrite").saveAsTable(f"{SLV}.slv_pricing")


# SILVER: PHARMACY NETWORK

In [0]:

# =========================
# 6) SILVER: PHARMACY NETWORK
# =========================
df_net_brz = spark.table(f"{BRZ}.brz_pharmacy_network")
df_net = (
    df_net_brz
    .transform(standardize_columns)
    .transform(add_ingest_metadata)
    .withColumn("contract_id", normalize_id("contract_id"))
    .withColumn("plan_id", normalize_id("plan_id", pad_left=3))
    .withColumn("segment_id", normalize_id("segment_id", pad_left=3))
    .withColumn("plan_bk", F.concat_ws("|", "contract_id", "plan_id", "segment_id"))
    .withColumn("pharmacy_number", normalize_id("pharmacy_number"))
    .withColumn("pharmacy_zipcode", normalize_id("pharmacy_zipcode", pad_left=5))
    .withColumn("preferred_status_retail", safe_bool_from_yn("preferred_status_retail"))
    .withColumn("preferred_status_mail", safe_bool_from_yn("preferred_status_mail"))
    .withColumn("pharmacy_retail", safe_bool_from_yn("pharmacy_retail"))
    .withColumn("pharmacy_mail", safe_bool_from_yn("pharmacy_mail"))
    .withColumn("in_area_flag", F.col("in_area_flag").cast("int"))
    .withColumn("is_in_area", F.when(F.col("in_area_flag") == 1, F.lit(True)).when(F.col("in_area_flag") == 0, F.lit(False)).otherwise(F.lit(None).cast("boolean")))
    .withColumn("floor_price", F.col("floor_price").cast(T.DecimalType(8,4)))
    # Dispensing fees (8,4)
    .withColumn("brand_dispensing_fee_30", F.col("brand_dispensing_fee_30").cast(T.DecimalType(8,4)))
    .withColumn("brand_dispensing_fee_60", F.col("brand_dispensing_fee_60").cast(T.DecimalType(8,4)))
    .withColumn("brand_dispensing_fee_90", F.col("brand_dispensing_fee_90").cast(T.DecimalType(8,4)))
    .withColumn("generic_dispensing_fee_30", F.col("generic_dispensing_fee_30").cast(T.DecimalType(8,4)))
    .withColumn("generic_dispensing_fee_60", F.col("generic_dispensing_fee_60").cast(T.DecimalType(8,4)))
    .withColumn("generic_dispensing_fee_90", F.col("generic_dispensing_fee_90").cast(T.DecimalType(8,4)))
    # Derived averages
    .withColumn("avg_brand_fee", (F.coalesce(F.col("brand_dispensing_fee_30"), F.lit(0)) +
                                 F.coalesce(F.col("brand_dispensing_fee_60"), F.lit(0)) +
                                 F.coalesce(F.col("brand_dispensing_fee_90"), F.lit(0))) / F.lit(3))
    .withColumn("avg_generic_fee", (F.coalesce(F.col("generic_dispensing_fee_30"), F.lit(0)) +
                                   F.coalesce(F.col("generic_dispensing_fee_60"), F.lit(0)) +
                                   F.coalesce(F.col("generic_dispensing_fee_90"), F.lit(0))) / F.lit(3))
)

# Add plan_key
if "plan_key" not in df_net.columns:
    df_net = df_net.join(df_plan_keys, on="plan_bk", how="left")

# Create network_key if absent
if "network_key" not in df_net.columns:
    df_net = df_net.withColumn("network_key", F.xxhash64("plan_key", "pharmacy_number"))

df_net.write.format("delta").mode("overwrite").saveAsTable(f"{SLV}.slv_pharmacy_network")


# SILVER: BASIC FORMULARY

In [0]:
# =========================
# 7) SILVER: BASIC FORMULARY
# =========================

df_form_brz = spark.table(f"{BRZ}.brz_basic_formulary")

df_form = (
    df_form_brz
    .transform(standardize_columns)
    .transform(add_ingest_metadata)
    .withColumn("formulary_id", normalize_id("formulary_id"))
    .withColumn("formulary_version", normalize_id("formulary_version"))
    .withColumn("contract_year", normalize_id("contract_year", pad_left=4))
    .withColumn("rxcui", normalize_id("rxcui", pad_left=8))
    .withColumn("ndc", normalize_id("ndc", pad_left=11))
    .withColumn("tier_level_value", F.col("tier_level_value").cast("int"))

    # --- FIX: normalize Y/N flags to integers 0/1
    .withColumn("quantity_limit_flag", flag_to_int("quantity_limit_yn"))
    .withColumn("prior_auth_flag", flag_to_int("prior_authorization_yn"))
    .withColumn("step_therapy_flag", flag_to_int("step_therapy_yn"))

    # keep original columns if you want, but DO NOT cast them to int
    .withColumn("quantity_limit_amount", F.trim(F.col("quantity_limit_amount").cast("string")))
    .withColumn("quantity_limit_days", F.col("quantity_limit_days").cast("int"))

    # derived
    .withColumn(
        "restriction_count",
        F.coalesce(F.col("quantity_limit_flag"), F.lit(0)) +
        F.coalesce(F.col("prior_auth_flag"), F.lit(0)) +
        F.coalesce(F.col("step_therapy_flag"), F.lit(0))
    )
    .withColumn(
        "formulary_drug_key",
        F.xxhash64("formulary_id", "formulary_version", "contract_year", "rxcui", "ndc")
    )
)

ins_ref = spark.table(f"{SLV}.slv_insulin_ndc_ref").selectExpr("ndc11 as ndc").dropDuplicates()
# form = spark.table(f"{SLV}.slv_basic_formulary")

form2 = (
    df_form
    .withColumn("ndc", F.col("ndc").cast("string"))   # adjust if your column is named differently
    .join(ins_ref.withColumn("is_insulin", F.lit(1)), on="ndc", how="left")
    .withColumn("is_insulin", F.coalesce(F.col("is_insulin"), F.lit(0)))
)

form2.write.format("delta").mode("overwrite").option("mergeSchema", "True").saveAsTable(f"{SLV}.slv_basic_formulary")


# SILVER: INSULIN COST

In [0]:
# =========================
# 8) SILVER: INSULIN COST
# =========================

df_insulin_cost_brz = spark.table(f"{BRZ}.brz_insulin_cost")

df_insulin_cost = (df_insulin_cost_brz
  .withColumn("plan_key", F.concat_ws("", F.col("CONTRACT_ID"), F.col("PLAN_ID"), F.col("SEGMENT_ID")))
  .withColumn("tier_raw", F.col("TIER"))
  .withColumn("tier", F.when(F.col("tier_raw") == ".", F.lit(None)).otherwise(F.col("tier_raw").cast("int")))
  .withColumn("tier_scope", F.when(F.col("tier_raw") == ".", F.lit("ALL")).otherwise(F.lit("tier_raw")))
  .withColumn("days_supply_label", F.col("DAYS_SUPPLY_LABEL"))
  .withColumn("copay_pref", F.col("copay_amt_pref_insln").cast("double"))
  .withColumn("copay_nonpref", F.col("copay_amt_nonpref_insln").cast("double"))
  .withColumn("copay_mail_pref", F.col("copay_amt_mail_pref_insln").cast("double"))
  .withColumn("copay_mail_nonpref", F.col("copay_amt_mail_nonpref_insln").cast("double"))
  .withColumn("insulin_min_copay_all_channels",
              F.least("copay_pref","copay_nonpref","copay_mail_pref","copay_mail_nonpref"))
  .withColumn("insulin_max_copay_all_channels",
              F.greatest("copay_pref","copay_nonpref","copay_mail_pref","copay_mail_nonpref"))
  .withColumn("min_insulin_copay", F.col("MIN_INSULIN_COPAY").cast("double"))
  .withColumn("max_insulin_copay", F.col("MAX_INSULIN_COPAY").cast("double"))
  .withColumn("exceeds_cap_flag", (F.col("EXCEEDS_CAP") == "Y").cast("int"))
)

df_insulin_cost.write.format("delta").mode("overwrite").saveAsTable(f"{SLV}.slv_plan_insulin_cost")

# display(df_insulin_cost)


# SILVER: EXCLUDED DRUGS

In [0]:
# =========================
# 9) SILVER: EXCLUDED DRUGS
# =========================

df_excluded_drugs_brz = spark.table(f"{BRZ}.brz_excluded_drugs")

plan_drug_access_rules = (df_excluded_drugs_brz
  .withColumn("plan_key", F.col("PLAN_KEY"))
  .withColumn("rxcui", normalize_id("rxcui", pad_left=8).cast("string"))
  .withColumn("pa", (F.col("PRIOR_AUTH_YN")=="Y").cast("int"))
  .withColumn("st", (F.col("STEP_THERAPY_YN")=="Y").cast("int"))
  .withColumn("ql", (F.col("QUANTITY_LIMIT_YN")=="Y").cast("int"))
  .withColumn("cap_benefit", (F.col("CAPPED_BENEFIT_YN")=="Y").cast("int"))
  .withColumn("completely_excluded", F.col("COMPLETELY_EXCLUDED").cast("boolean"))
  .withColumn("conditionally_covered", F.col("CONDITIONALLY_COVERED").cast("boolean"))
  .withColumn(
      "drug_access_status",
      F.when(F.col("completely_excluded")==True, F.lit("EXCLUDED"))
       .when(F.col("conditionally_covered")==True, F.lit("CONDITIONAL"))
       .otherwise(F.lit("COVERED"))
  )
  .withColumn("restriction_score", F.col("pa")+F.col("st")+F.col("ql")+F.col("cap_benefit"))
)

plan_drug_access_rules.write.format("delta").mode("overwrite").saveAsTable(f"{SLV}.slv_plan_drug_access_rules")


# SILVER: DRUG REF

In [0]:
!pip install pdfplumber

In [0]:
# Databricks notebook cell

# 1) Download the CMS PDF (choose one canonical version; 2023 shown here)
pdf_url = "https://www.cms.gov/priorities/innovation/media/document/partd-seniorsav-ndclist-2023"
pdf_path = "/Volumes/cms_partd_bronze/raw/raw/cms_insulin_ndc_list_2023.pdf"

import requests
r = requests.get(pdf_url, timeout=60)
r.raise_for_status()
with open(pdf_path, "wb") as f:
    f.write(r.content)

# 2) Extract candidate NDC strings from PDF text
import re
import pdfplumber

ndc_pattern = re.compile(r"\b\d{4,5}-\d{3,4}-\d{1,2}\b")  # 10/11-digit formatted NDCs

ndcs = set()
with pdfplumber.open(pdf_path) as pdf:
    for page in pdf.pages:
        text = page.extract_text() or ""
        for m in ndc_pattern.findall(text):
            ndcs.add(m)

# 3) Create Spark DF and normalize NDC to 11-digit "ndc11"
from pyspark.sql import functions as F
ndc_rows = [(x,) for x in sorted(ndcs)]
df = spark.createDataFrame(ndc_rows, ["ndc_formatted"])

# Simple NDC11 normalizer for formatted NDCs (labeler code - product code - package code)
# Pads each segment to 5-4-2 and concatenates.
def to_ndc11(col):
    parts = F.split(col, "-")
    return F.concat(
        F.lpad(parts.getItem(0), 5, "0"),
        F.lpad(parts.getItem(1), 4, "0"),
        F.lpad(parts.getItem(2), 2, "0"),
    )

df = (
    df
    .withColumn("ndc11", to_ndc11(F.col("ndc_formatted")))
    .withColumn("source", F.lit("CMS Part D Senior Savings Model NDC List (CY2023)"))
    .withColumn("source_year", F.lit(2023))
    .withColumn("ingested_ts", F.current_timestamp())
    .dropDuplicates(["ndc11"])
)

df.write.format("delta").mode("overwrite").saveAsTable(f"{SLV}.slv_insulin_ndc_ref")


In [0]:
# Requires a reference table that contains ndc11 and rxcui
drug_ref = spark.table(f"{SLV}.slv_basic_formulary") \
    .select(F.col("ndc").alias("ndc11"), F.col("rxcui").cast("string").alias("rxcui")) \
    .dropDuplicates(["ndc11","rxcui"])

ins_ndc = spark.table(f"{SLV}.slv_insulin_ndc_ref").select("ndc11","source","source_year")

slv_insulin_drug_ref = (
    ins_ndc
    .join(drug_ref, on="ndc11", how="left")
    .withColumn("is_insulin", F.lit(1))
    .withColumn("ref_ts", F.current_timestamp())
)

slv_insulin_drug_ref = slv_insulin_drug_ref.withColumnRenamed("ndc11", "ndc")

# Optional: quality checks
# - How many insulin NDCs map to a RxCUI?
# - Which NDCs do not map (missing in your ref)?
slv_insulin_drug_ref.write.format("delta").saveAsTable(f"{SLV}.slv_insulin_drug_ref")


In [0]:
%sql
drop table cms_partd_silver.core.slv_insulin_drug_ref

In [0]:
# =========================
# 8) QUICK SANITY CHECKS
# =========================
print("Silver tables created:")
for t in ["slv_plan", "slv_geo", "slv_beneficiary_cost", "slv_pricing", "slv_pharmacy_network", "slv_basic_formulary"]:
    print(" -", f"{SLV}.{t}", "rows =", spark.table(f"{SLV}.{t}").count())